In [1]:
from lxml import etree as ET
#Not sure if I'll need:
#import subprocess
#from pathlib import Path

Section number

In [15]:
sec_num='9-4'

Input and Output files

In [16]:
input = '../source/section-' + sec_num + '.ptx'
output = '../source/slides-' + sec_num + '.ptx'

Parse the section file and the docinfo as xml trees

In [17]:
section_tree = ET.parse(input)
section = section_tree.getroot()
section.tag = 'slideshow'
section.attrib.clear()
docinfo_tree = ET.parse('../source/meta_docinfo.ptx')
docinfo = docinfo_tree.getroot()
#ET.tostring(section_tree)
#ET.tostring(docinfo_tree)

Create the new xml tree and isert the previous trees

In [18]:
slides = ET.Element('pretext')
slides.insert(0, docinfo)
slides.append(section)
slides_tree = ET.ElementTree(slides)
#ET.tostring(slides)

Add subtitle, frontmatter, and jesus creed

In [19]:
title = slides.find('.//title')
title.text = sec_num + ' : ' + title.text
subtitle = ET.parse('subtitle.txt').getroot()
front_matter = ET.parse('front_matter.txt').getroot()
jesus_creed = ET.parse('jesus-creed.txt').getroot()
title.addnext(subtitle)
subtitle.addnext(front_matter)
front_matter.addnext(jesus_creed)
#ET.tostring(slides)
print(title.text)

9-4 : Chi-square goodness-of-fit test


Make a slide for every activity (exerise).  For exercises that break into tasks split into a slide for every task.

In [20]:
for num, block in enumerate(slides.iter(tag=('activity'))):
    if block.find('task') is not None:
        for intro in block.iter('introduction'):
            intro.tag = 'slide'
            if block.find('title') is not None:
                slide_title = ET.Element('title') 
                slide_title = block.find('title')
                slide_title.text = 'block ' + str(num + 1) + ': ' + slide_title.text
            else:
                slide_title = ET.Element('title') 
                slide_title.text = 'Exercise ' + str(num + 1)
            intro.insert(0,slide_title)
        for a, task in enumerate(block.iter('task')):
            ET.strip_tags(task, 'statement')
            ET.strip_elements(task, 'solution')
            slide_title = ET.Element('title') 
            slide_title.text = 'Exercise ' + str(num + 1) + chr(a + 97) + ': '
            task.insert(0, slide_title)
            task.tag = 'slide'
        for concl in block.iter('conclusion'):
            concl.tag = 'slide'
    else:
        block.tag = 'slide'
        ET.strip_tags(block, 'statement')
        ET.strip_elements(block, 'solution')
        slide_title = ET.Element('title')
        slide_title.text = 'Exercise ' + str(num + 1)
        block.insert(0,slide_title)

#print(ET.tostring(slides))
#When exercises have tasks, we still need to unwrap the original exercises
ET.strip_tags(slides, 'activity')

Make a slide for every block.  That is, every definition, remark, insight, convention, figure, or table.

In [21]:
for num, block in enumerate(slides.iter(tag=('definition','remark', 'insight', 'convention', 'figure', 'table'))):
    slide = ET.Element('slide')
    block.addprevious(slide)
    slide.insert(0,block)
    slide_title = ET.Element('title') 
    slide_title.text = block.tag.title() + ' ' + str(num + 1) + ': '
    slide.insert(0,slide_title)

In [22]:
#ET.tostring(slides)

In [23]:
for subsec in slides.iter('exercises'):
    for p in subsec.findall('p'):
        print(p.text)

Change all subsections to sections, and wrap loose paragraphs and images in slides

In [24]:
for subsec in slides.iter('subsection', 'slideshow'):
    if subsec.tag == 'subsection':
        subsec.tag = 'section'
    for p in subsec.findall('p') + subsec.findall('image'):
        slide = ET.Element('slide')
        p.addprevious(slide)
        slide.insert(0,p)

Write to a file

In [25]:
ET.indent(slides_tree)
slides_tree.write(output, encoding='UTF-8', xml_declaration=True)
#??slides_tree.write

In [26]:
#ET.tostring(slides)

In [27]:
for url in slides.iter('url'):
    print(url.attrib)

{'href': 'https://static-content.zybooks.com/applets/2021/gof/GOF.html?hideExtras=1'}
